In [1]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import PeftModel
import torch
torch.cuda.empty_cache()
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secret = UserSecretsClient()
login(token=secret.get_secret("hugging face"))


BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
HF_REPO    = "dhrv-dng/qwen-toxic-classifier-multilingual"

tokenizer = AutoTokenizer.from_pretrained(HF_REPO, trust_remote_code=True)

base_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    torch_dtype=torch.float32,
    device_map="auto",
    trust_remote_code=True,
    ignore_mismatched_sizes=True,
)
base_model.config.pad_token_id = tokenizer.pad_token_id

inf_model = PeftModel.from_pretrained(base_model, HF_REPO)

tokenizer_config.json:   0%|          | 0.00/692 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/4.39M [00:00<?, ?B/s]

In [2]:
def predict(text: str) -> int:
    """Return 1 if toxic, 0 if non-toxic."""
    prompt = f"{SYSTEM}\nClassify: {text}"
    enc = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LEN,
    ).to(inf_model.device)
    with torch.no_grad():
        logits = inf_model(**enc).logits
    return int(logits.argmax(-1).item())

In [6]:
from datasets import load_dataset

dataset = load_dataset(
    "csv",
    data_files={
        "test":  "/kaggle/input/datasets/dhrvdng/multilingual-val/val_en_hi_ru.csv",
    }
)
test_dataset  = dataset["test"]
test_dataset = test_dataset.rename_column("label", "toxic")

Generating test split: 0 examples [00:00, ? examples/s]

In [3]:
SYSTEM = "You are a multilingual binary toxicity classifier understanding hindi, hinglish, english and russian."
MAX_LEN = 128

In [6]:
# Metrics

import evaluate
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

accuracy_metric = evaluate.load("accuracy")
f1_metric       = evaluate.load("f1")
 
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc   = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1    = f1_metric.compute(predictions=preds, references=labels, average="weighted")["f1"]
    f1_tox = f1_metric.compute(predictions=preds, references=labels, average="binary")["f1"]
    return {"accuracy": acc, "f1_weighted": f1, "f1_toxic": f1_tox}

In [7]:
def predict_batch(texts: list, batch_size: int = 32) -> list:
    preds = []
    for i in range(0, len(texts), batch_size):
        batch   = texts[i : i + batch_size]
        prompts = [f"{SYSTEM}\nClassify: {t}" for t in batch]  # match your existing prompt format
        enc = tokenizer(
            prompts,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_LEN,
            padding=True,
        ).to(inf_model.device)   # matches your existing variable name
        with torch.no_grad():
            logits = inf_model(**enc).logits
        preds.extend(logits.argmax(-1).cpu().tolist())

        if (i // batch_size) % 10 == 0:
            print(f"{i}/{len(texts)} done...")
    return preds

# Evaluation
texts  = test_dataset["text"]
y_true = [int(x) for x in test_dataset["toxic"]]
y_pred = predict_batch(texts, batch_size=32)

print(f"Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
print(f"Precision : {precision_score(y_true, y_pred):.4f}")
print(f"Recall    : {recall_score(y_true, y_pred):.4f}")
print(f"F1-score  : {f1_score(y_true, y_pred):.4f}")
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, digits=4))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_true, y_pred))

NameError: name 'test_dataset' is not defined

In [10]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)

datasets = {
    "English": "/kaggle/input/datasets/dhrvdng/eval-lang/textdetox_en_sample_168.csv",
    "Russian": "/kaggle/input/datasets/dhrvdng/eval-lang/textdetox_ru_sample_168.csv",
    "Hindi":   "/kaggle/input/datasets/dhrvdng/eval-lang/textdetox_hi_sample_168.csv",
}

dfs = []
for lang, path in datasets.items():
    df = pd.read_csv(path)
    df["language"] = lang   # tag each row with its language
    dfs.append(df)

combined_df = pd.concat(dfs, ignore_index=True)
print(f"Combined dataset size: {len(combined_df)}")   # should print 504

results = []

for lang, path in datasets.items():
    df     = pd.read_csv(path)
    texts  = df["text"].tolist()
    y_true = [1 if x >= 0.5 else 0 for x in df["toxic"].tolist()]
    
    y_pred = predict_batch(texts, batch_size=32)
    
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    
    print(f"\n{'='*50}")
    print(f"  {lang}  |  Samples: {len(df)}")
    print(f"{'='*50}")
    print(f"Accuracy  : {acc:.4f}")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"F1-score  : {f1:.4f}")
    print(f"\n{classification_report(y_true, y_pred, target_names=['non-toxic', 'toxic'], digits=4)}")
    print(f"Confusion Matrix:\n{confusion_matrix(y_true, y_pred)}")
    
    results.append({
        "language":  lang,
        "samples":   len(df),
        "accuracy":  acc,
        "precision": prec,
        "recall":    rec,
        "f1":        f1,
    })

# Summary table
print(f"\n{'='*60}")
print("  SUMMARY BY LANGUAGE")
print(f"{'='*60}")
print(f"{'Language':<12} {'Samples':>8} {'Accuracy':>9} {'Precision':>10} {'Recall':>8} {'F1':>8}")
print("-" * 60)
for r in results:
    print(
        f"{r['language']:<12} "
        f"{r['samples']:>8} "
        f"{r['accuracy']:>9.4f} "
        f"{r['precision']:>10.4f} "
        f"{r['recall']:>8.4f} "
        f"{r['f1']:>8.4f}"
    )

Combined dataset size: 504
0/168 done...

  English  |  Samples: 168
Accuracy  : 0.9881
Precision : 0.9881
Recall    : 0.9881
F1-score  : 0.9881

              precision    recall  f1-score   support

   non-toxic     0.9881    0.9881    0.9881        84
       toxic     0.9881    0.9881    0.9881        84

    accuracy                         0.9881       168
   macro avg     0.9881    0.9881    0.9881       168
weighted avg     0.9881    0.9881    0.9881       168

Confusion Matrix:
[[83  1]
 [ 1 83]]
0/168 done...

  Russian  |  Samples: 168
Accuracy  : 0.9464
Precision : 0.9630
Recall    : 0.9286
F1-score  : 0.9455

              precision    recall  f1-score   support

   non-toxic     0.9310    0.9643    0.9474        84
       toxic     0.9630    0.9286    0.9455        84

    accuracy                         0.9464       168
   macro avg     0.9470    0.9464    0.9464       168
weighted avg     0.9470    0.9464    0.9464       168

Confusion Matrix:
[[81  3]
 [ 6 78]]
0/168 d

In [11]:
# Run evaluation on the full combined set
texts  = combined_df["text"].tolist()
y_true = [1 if x >= 0.5 else 0 for x in combined_df["toxic"].tolist()]

y_pred = predict_batch(texts, batch_size=32)

acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
rec  = recall_score(y_true, y_pred, zero_division=0)
f1   = f1_score(y_true, y_pred, zero_division=0)

print(f"\n{'='*50}")
print(f"  COMBINED MULTILINGUAL  |  Samples: {len(combined_df)}")
print(f"{'='*50}")
print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1-score  : {f1:.4f}")

print(f"\n{classification_report(y_true, y_pred, target_names=['non-toxic', 'toxic'], digits=4)}")
print(f"Confusion Matrix:\n{confusion_matrix(y_true, y_pred)}")

0/504 done...
320/504 done...

  COMBINED MULTILINGUAL  |  Samples: 504
Accuracy  : 0.9107
Precision : 0.9641
Recall    : 0.8532
F1-score  : 0.9053

              precision    recall  f1-score   support

   non-toxic     0.8683    0.9683    0.9156       252
       toxic     0.9641    0.8532    0.9053       252

    accuracy                         0.9107       504
   macro avg     0.9162    0.9107    0.9104       504
weighted avg     0.9162    0.9107    0.9104       504

Confusion Matrix:
[[244   8]
 [ 37 215]]
